# Contrôle qualité du pipeline

Vérifie les **données réelles** après une exécution du pipeline (`pipeline_bio`), couche par couche,
et enregistre chaque résultat dans `audit.controle_qualite`.

Les tests de la CI (`sql/tests/test_init.sql`) vérifient la **structure** de la base sur des données fictives ;
ce notebook vérifie le **contenu** : peut-on faire confiance à ce que Power BI va afficher ?

| Statut | Sens |
|---|---|
| `OK` | conforme au seuil |
| `ALERTE` | à examiner ; les données restent utilisables |
| `ECHEC` | données fausses : le notebook s'arrête en erreur (et la tâche Airflow échoue) |

**Connexion :** rôle `qa_reader` (lecture de toutes les couches, écriture dans `audit.controle_qualite` seulement).

**Lancement :** cellule par cellule dans VS Code, ou en ligne de commande avec papermill :

```
papermill notebooks/qualite/controle_qualite.ipynb sortie.ipynb -p date_copie 2026-11-08
```

## 1. Paramètres

Cellule étiquetée `parameters` : papermill y injecte les valeurs passées avec `-p`.
Sans date, le notebook contrôle la dernière copie de l'annuaire chargée en bronze.

In [1]:
# Copie de l'annuaire à contrôler (AAAA-MM-JJ). None = la dernière copie présente en bronze.
date_copie = None

## 2. Connexion

Les réglages viennent des **variables d'environnement** (dans le conteneur Airflow), sinon du fichier `.env`
du projet (sur le PC). Le mot de passe n'est jamais écrit dans le notebook.

In [2]:
import os
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
import psycopg

NOTEBOOK = "qualite/controle_qualite.ipynb"

# Racine du projet : premier dossier parent qui contient docker-compose.yml
RACINE = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yml").exists()),
    None,
)


def lire_env(chemin):
    """Lit un fichier .env (lignes CLE=valeur) dans un dictionnaire."""
    valeurs = {}
    if chemin is not None and chemin.exists():
        for ligne in chemin.read_text(encoding="utf-8").splitlines():
            ligne = ligne.strip()
            if ligne and not ligne.startswith("#") and "=" in ligne:
                cle, valeur = ligne.split("=", 1)
                valeurs[cle.strip()] = valeur.strip()
    return valeurs


ENV = lire_env(RACINE / ".env" if RACINE else None)


def reglage(cle_env, cle_fichier, defaut=None):
    """Variable d'environnement, sinon valeur du fichier .env, sinon valeur par défaut."""
    return os.environ.get(cle_env) or ENV.get(cle_fichier) or defaut


# 127.0.0.1 et non localhost : sous Windows, localhost désigne d'abord l'adresse IPv6 ::1,
# sur laquelle PostgreSQL n'écoute pas (port publié sur 127.0.0.1 seulement).
# connect_timeout : sans délai maximal, une connexion impossible attendrait indéfiniment.
CONNEXION = {
    "host": reglage("PG_HOTE", "PG_HOTE", "127.0.0.1"),
    "port": reglage("PG_PORT", "POSTGRES_PORT", "5432"),
    "dbname": reglage("PG_BASE", "POSTGRES_DB", "bio_observatoire"),
    "user": "qa_reader",
    "password": reglage("QA_READER_PASSWORD", "QA_READER_PASSWORD"),
    "connect_timeout": 10,
}

with psycopg.connect(**CONNEXION) as cnx:
    print(cnx.execute("SELECT current_user, current_database()").fetchone())

('qa_reader', 'bio_observatoire')


## 3. Copie contrôlée et exécution associée

Les résultats sont rattachés à l'exécution de `silver_annuaire` qui a traité la copie
(clé étrangère vers `audit.execution`) : on sait ainsi quel chargement a été contrôlé.

In [3]:
def valeur(sql, parametres=None):
    """Exécute une requête qui renvoie une seule valeur."""
    with psycopg.connect(**CONNEXION) as cnx:
        return cnx.execute(sql, parametres or {}).fetchone()[0]


if date_copie is None:
    date_copie = str(valeur("SELECT max(_date_copie) FROM bronze.agence_bio_operateur"))

execution_id = valeur(
    """
    SELECT max(execution_id) FROM audit.execution
    WHERE job_nom = 'silver_annuaire' AND statut = 'SUCCES' AND date_copie = %(date)s
    """,
    {"date": date_copie},
)
print(f"Copie contrôlée : {date_copie} ; exécution silver_annuaire : {execution_id}")

Copie contrôlée : 2026-11-08 ; exécution silver_annuaire : 108


## 4. Les contrôles

Chaque contrôle est une requête SQL qui renvoie **une valeur**, comparée à un **seuil** :

- `max` : la valeur doit être inférieure ou égale au seuil (ex. 0 doublon) ;
- `min` : la valeur doit être supérieure ou égale au seuil (ex. au moins 98 % de SIRET valides).

Si la règle n'est pas respectée, le statut est le **niveau** du contrôle (`ALERTE` ou `ECHEC`).
`%(date)s` est remplacé par la date de la copie contrôlée.

In [4]:
@dataclass
class Controle:
    couche: str
    nom: str
    sql: str
    regle: str      # "max" ou "min"
    seuil: float
    niveau: str     # statut si la règle n'est pas respectée : "ALERTE" ou "ECHEC"
    sens: str       # ce que mesure le contrôle, enregistré en commentaire


CONTROLES = [
    # --- Bronze : la copie est-elle arrivée, et a-t-elle une taille normale ?
    Controle(
        "bronze", "copie_presente",
        "SELECT count(*) FROM bronze.agence_bio_operateur WHERE _date_copie = %(date)s",
        "min", 1, "ECHEC",
        "Lignes de la copie en bronze",
    ),
    Controle(
        "bronze", "variation_volume_pct",
        """
        WITH copies AS (
            SELECT _date_copie AS d, count(*) AS n FROM bronze.agence_bio_operateur
            WHERE _date_copie <= %(date)s GROUP BY 1 ORDER BY 1 DESC LIMIT 2
        )
        SELECT coalesce(abs(max(n) FILTER (WHERE d = %(date)s) - max(n) FILTER (WHERE d < %(date)s))
                        * 100.0 / max(n) FILTER (WHERE d < %(date)s), 0)
        FROM copies
        """,
        "max", 2, "ALERTE",
        "Variation du nombre de lignes par rapport à la copie précédente, en %",
    ),
    # --- Silver : qualité du nettoyage et cohérence de l'historique
    Controle(
        "silver", "copie_traitee",
        "SELECT count(*) FROM silver.operateur WHERE date_derniere_observation = %(date)s",
        "min", 1, "ECHEC",
        "Opérateurs observés dans la copie : silver a bien traité cette copie",
    ),
    Controle(
        "silver", "siret_valide_pct",
        """
        SELECT 100.0 * count(*) FILTER (WHERE siret_statut = 'VALIDE') / nullif(count(*), 0)
        FROM silver.operateur WHERE est_present
        """,
        "min", 98, "ALERTE",
        "Part des opérateurs présents avec un SIRET valide, en %",
    ),
    Controle(
        "silver", "operateurs_non_rattaches",
        "SELECT count(*) FROM silver.operateur WHERE est_present AND code_departement = '00'",
        "max", 50, "ALERTE",
        "Opérateurs présents rattachés au département inconnu (00)",
    ),
    Controle(
        "silver", "versions_courantes_en_double",
        """
        SELECT count(*) FROM (
            SELECT 1 FROM silver.certification WHERE est_courant
            GROUP BY numero_bio, certificateur_id HAVING count(*) > 1
        ) AS doublons
        """,
        "max", 0, "ECHEC",
        "Certifications avec plus d'une version en cours (SCD 2)",
    ),
    Controle(
        "silver", "versions_chevauchantes",
        """
        SELECT count(*)
        FROM silver.certification a
        JOIN silver.certification b
          ON b.numero_bio = a.numero_bio
         AND b.certificateur_id = a.certificateur_id
         AND b.certification_version_id > a.certification_version_id
        WHERE a.date_debut < coalesce(b.date_fin, 'infinity')
          AND b.date_debut < coalesce(a.date_fin, 'infinity')
        """,
        "max", 0, "ECHEC",
        "Paires de versions dont les intervalles [date_debut, date_fin[ se recouvrent (SCD 2)",
    ),
    Controle(
        "silver", "presence_incoherente",
        """
        SELECT count(*) FROM silver.operateur o
        WHERE o.est_present <> EXISTS (
            SELECT 1 FROM silver.certification c WHERE c.numero_bio = o.numero_bio AND c.est_courant
        )
        """,
        "max", 0, "ECHEC",
        "Opérateurs présents sans certification en cours, ou absents avec une certification en cours",
    ),
    Controle(
        "silver", "departements_sans_population",
        """
        SELECT count(*) FROM silver.ref_departement d
        WHERE d.code_departement NOT IN ('99', '00')
          AND NOT EXISTS (SELECT 1 FROM silver.population_departement p
                          WHERE p.code_departement = d.code_departement)
        """,
        "max", 1, "ALERTE",
        "Départements sans population (Mayotte est absente du millésime 2023)",
    ),
    Controle(
        "silver", "rejets_numero_bio",
        """
        SELECT count(*) FROM audit.rejet r
        JOIN audit.execution e USING (execution_id)
        WHERE r.motif_code = 'NUMERO_BIO_ABSENT' AND e.date_copie = %(date)s AND e.statut = 'SUCCES'
        """,
        "max", 0, "ALERTE",
        "Lignes de la copie rejetées faute de numéro bio",
    ),
    # --- Gold : l'étoile reflète-t-elle exactement silver ?
    Controle(
        "gold", "ecart_faits_silver",
        """
        SELECT abs((SELECT count(*) FROM gold.fait_certification_activite)
                 - (SELECT count(*) FROM silver.certification_activite a
                    JOIN silver.certification c USING (certification_version_id)
                    WHERE c.est_courant))
        """,
        "max", 0, "ECHEC",
        "Écart entre les faits de gold et les activités des certifications en cours de silver",
    ),
    Controle(
        "gold", "ecart_operateurs_silver",
        """
        SELECT abs((SELECT count(DISTINCT operateur_id) FROM gold.fait_certification_activite)
                 - (SELECT count(*) FROM silver.operateur WHERE est_present))
        """,
        "max", 0, "ALERTE",
        "Écart entre les opérateurs de gold et les opérateurs présents de silver",
    ),
    Controle(
        "gold", "faits_membre_inconnu",
        """
        SELECT count(*) FROM gold.fait_certification_activite
        WHERE certificateur_id = -1 OR activite_id = -1 OR departement_id = -1
        """,
        "max", 0, "ALERTE",
        "Faits rattachés au membre -1 d'une dimension (hors date d'engagement)",
    ),
]
print(f"{len(CONTROLES)} contrôles définis")

13 contrôles définis


## 5. Exécution des contrôles

In [5]:
def statut(controle, mesure):
    """OK si la règle est respectée, sinon le niveau du contrôle."""
    if mesure is None:
        return controle.niveau
    respecte = mesure <= controle.seuil if controle.regle == "max" else mesure >= controle.seuil
    return "OK" if respecte else controle.niveau


resultats = []
for c in CONTROLES:
    mesure = valeur(c.sql, {"date": date_copie})
    mesure = None if mesure is None else round(float(mesure), 4)
    resultats.append({
        "couche": c.couche,
        "controle": c.nom,
        "valeur": mesure,
        "regle": f"{'≤' if c.regle == 'max' else '≥'} {c.seuil:g}",
        "statut": statut(c, mesure),
        "sens": c.sens,
    })

resultats = pd.DataFrame(resultats)
resultats

,couche,controle,valeur,regle,statut,sens
0,bronze,copie_presente,88153.0000,≥ 1,OK,Lignes de la copie en bronze
1,bronze,variation_volume_pct,0.0023,≤ 2,OK,Variation du nombre de lignes par rapport à la...
2,silver,copie_traitee,88084.0000,≥ 1,OK,Opérateurs observés dans la copie : silver a b...
3,silver,siret_valide_pct,99.1429,≥ 98,OK,Part des opérateurs présents avec un SIRET val...
4,silver,operateurs_non_rattaches,2.0000,≤ 50,OK,Opérateurs présents rattachés au département i...
5,silver,versions_courantes_en_double,0.0000,≤ 0,OK,Certifications avec plus d'une version en cour...
6,silver,versions_chevauchantes,0.0000,≤ 0,OK,Paires de versions dont les intervalles [date_...
7,silver,presence_incoherente,0.0000,≤ 0,OK,Opérateurs présents sans certification en cour...
8,silver,departements_sans_population,1.0000,≤ 1,OK,Départements sans population (Mayotte est abse...
9,silver,rejets_numero_bio,0.0000,≤ 0,OK,Lignes de la copie rejetées faute de numéro bio


## 6. Enregistrement dans `audit.controle_qualite`

Une seule transaction : soit tous les résultats sont enregistrés, soit aucun.

In [6]:
with psycopg.connect(**CONNEXION) as cnx:
    with cnx.cursor() as cur:
        cur.executemany(
            """
            INSERT INTO audit.controle_qualite
                (execution_id, couche, notebook, nom_controle, valeur_mesuree, seuil, statut, commentaire)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
            """,
            [
                (execution_id, c.couche, NOTEBOOK, c.nom, r.valeur, c.seuil, r.statut,
                 f"Copie {date_copie} : {c.sens}")
                for c, r in zip(CONTROLES, resultats.itertuples(), strict=True)
            ],
        )
print(f"{len(resultats)} résultats enregistrés")

13 résultats enregistrés


## 7. Bilan

Un `ECHEC` arrête le notebook en erreur : lancé par papermill dans Airflow, il fait échouer la tâche,
ce qui se voit en rouge dans l'interface. Une `ALERTE` est seulement signalée.

In [7]:
bilan = resultats["statut"].value_counts()
print(bilan.to_string())

alertes = resultats[resultats["statut"] == "ALERTE"]
if not alertes.empty:
    print("\nAlertes à examiner :")
    print(alertes[["couche", "controle", "valeur", "regle"]].to_string(index=False))

echecs = resultats[resultats["statut"] == "ECHEC"]
if not echecs.empty:
    raise RuntimeError(f"{len(echecs)} contrôle(s) en échec : {', '.join(echecs['controle'])}")

statut
OK    13
